In [ ]:
!free -h | head -2
!df -h /content | tail -1

               total        used        free      shared  buff/cache   available
Mem:            12Gi       763Mi       8.9Gi       2.0Mi       3.0Gi        11Gi
overlay         108G   20G   88G  19% /


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORK = '/content/drive/MyDrive/market_gap'
os.makedirs(WORK, exist_ok=True)

Mounted at /content/drive


In [ ]:
!wget -c https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz -O /content/off.csv.gz
!ls -lh /content/off.csv.gz

--2026-07-17 12:32:32--  https://static.openfoodfacts.org/data/en.openfoodfacts.org.products.csv.gz
Resolving static.openfoodfacts.org (static.openfoodfacts.org)... 151.115.132.10
Connecting to static.openfoodfacts.org (static.openfoodfacts.org)|151.115.132.10|:443... connected.
HTTP request sent, awaiting response... 302 Moved Temporarily
Location: https://openfoodfacts-ds.s3.eu-west-3.amazonaws.com/en.openfoodfacts.org.products.csv.gz [following]
--2026-07-17 12:32:33--  https://openfoodfacts-ds.s3.eu-west-3.amazonaws.com/en.openfoodfacts.org.products.csv.gz
Resolving openfoodfacts-ds.s3.eu-west-3.amazonaws.com (openfoodfacts-ds.s3.eu-west-3.amazonaws.com)... 3.5.204.17, 3.5.206.249
Connecting to openfoodfacts-ds.s3.eu-west-3.amazonaws.com (openfoodfacts-ds.s3.eu-west-3.amazonaws.com)|3.5.204.17|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1275171186 (1.2G) [application/gzip]
Saving to: ‘/content/off.csv.gz’

/content/off.csv.gz 100%[===================>]

In [ ]:
import pandas as pd

head = pd.read_csv('/content/off.csv.gz', sep='\t', nrows=5,
                   on_bad_lines='skip', low_memory=False)
print(head.shape)
print([c for c in head.columns if 'sugar' in c or 'protein' in c or 'categor' in c])

(5, 211)
['categories', 'categories_tags', 'categories_en', 'main_category', 'main_category_en', 'sugars_100g', 'added-sugars_100g', 'proteins_100g', 'serum-proteins_100g', 'collagen-meat-protein-ratio_100g']


In [ ]:
cols = ['code','product_name','brands','countries_en','categories_tags',
        'ingredients_text','energy-kcal_100g','fat_100g','saturated-fat_100g',
        'sugars_100g','fiber_100g','proteins_100g','salt_100g','nutriscore_grade']

missing = [c for c in cols if c not in head.columns]
print("Missing:", missing)
cols = [c for c in cols if c in head.columns]

Missing: []


In [ ]:
CHUNK = 200_000
chunks, scanned = [], 0

reader = pd.read_csv('/content/off.csv.gz', sep='\t', usecols=cols,
                     chunksize=CHUNK, on_bad_lines='skip',
                     low_memory=False, encoding_errors='replace')

for i, chunk in enumerate(reader):
    scanned += len(chunk)
    mask = chunk['categories_tags'].fillna('').str.contains('en:snacks', case=False, na=False)
    hit = chunk[mask]
    if len(hit):
        chunks.append(hit)
    print(f"chunk {i:>2} | scanned {scanned:>9,} | kept so far {sum(len(c) for c in chunks):>7,}")

snacks = pd.concat(chunks, ignore_index=True)
del chunks
print("\nFinal:", snacks.shape)

chunk  0 | scanned   200,000 | kept so far  23,316
chunk  1 | scanned   400,000 | kept so far  42,713
chunk  2 | scanned   600,000 | kept so far  64,041
chunk  3 | scanned   800,000 | kept so far  71,432
chunk  4 | scanned 1,000,000 | kept so far  90,624
chunk  5 | scanned 1,200,000 | kept so far 109,729
chunk  6 | scanned 1,400,000 | kept so far 119,492
chunk  7 | scanned 1,600,000 | kept so far 122,878
chunk  8 | scanned 1,800,000 | kept so far 141,803
chunk  9 | scanned 2,000,000 | kept so far 157,968
chunk 10 | scanned 2,200,000 | kept so far 175,336
chunk 11 | scanned 2,400,000 | kept so far 190,011
chunk 12 | scanned 2,600,000 | kept so far 201,798
chunk 13 | scanned 2,800,000 | kept so far 210,837
chunk 14 | scanned 3,000,000 | kept so far 224,031
chunk 15 | scanned 3,200,000 | kept so far 236,933
chunk 16 | scanned 3,400,000 | kept so far 246,010
chunk 17 | scanned 3,600,000 | kept so far 264,602
chunk 18 | scanned 3,800,000 | kept so far 285,448
chunk 19 | scanned 4,000,000 | 

In [ ]:
print(snacks.shape)
print(f"Memory: {snacks.memory_usage(deep=True).sum()/1e6:.0f} MB")
snacks[['product_name','categories_tags','sugars_100g','proteins_100g']].head(10)

(331907, 14)
Memory: 283 MB


,product_name,categories_tags,sugars_100g,proteins_100g
0,xxx,"en:beverages-and-beverages-preparations,en:bev...",NaN,NaN
1,Powdered peanut butter,"en:snacks,en:meals,en:rice-dishes,en:risottos,...",NaN,NaN
2,Madeleines ChocoLait,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,NaN
3,Nesquik moins de sucre,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,NaN
4,Farandole de madeleine,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,NaN
5,Multi Patents Collagen Peptides,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,NaN
6,Madeleines,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,NaN
7,Chunky blue cheese dressing,"en:snacks,en:condiments,en:sweet-snacks,en:bis...",NaN,NaN
8,Calcium,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",NaN,NaN
9,Vitamine C poudre,"en:snacks,en:sweet-snacks,en:cocoa-and-its-pro...",NaN,NaN


In [ ]:
from collections import Counter
tags = Counter()
for t in snacks['categories_tags'].dropna().head(50_000):
    tags.update(x.strip() for x in t.split(','))
pd.Series(dict(tags.most_common(30)))

,0
en:snacks,49978
en:sweet-snacks,29873
en:confectioneries,14476
en:biscuits-and-cakes,13182
en:biscuits,6496
en:salty-snacks,4868
en:cocoa-and-its-products,4666
en:appetizers,4472
en:cakes,4258
en:chocolate-candies,3828


In [ ]:
snacks['countries_en'].value_counts().head(10)
snacks[['sugars_100g','proteins_100g','product_name']].isna().mean()

,0
sugars_100g,0.409383
proteins_100g,0.404866
product_name,0.032503


In [ ]:
snacks['code'] = snacks['code'].astype(str)

# catch any other mixed-type object columns before they bite
for c in snacks.select_dtypes('object').columns:
    snacks[c] = snacks[c].astype(str).replace('nan', pd.NA)

snacks.to_parquet(f'{WORK}/snacks_raw.parquet', index=False)
!ls -lh "{WORK}/snacks_raw.parquet"

-rw------- 1 root root 42M Jul 17 13:07 /content/drive/MyDrive/market_gap/snacks_raw.parquet


In [ ]:
snacks['countries_en'].value_counts().head(10)

,count
countries_en,
France,88462
United States,88382
Italy,17668
Germany,15952
Spain,15812
United Kingdom,10713
Switzerland,5630
Belgium,5333
Canada,4874


In [ ]:
snacks['has_nutrition'] = snacks['sugars_100g'].notna() & snacks['proteins_100g'].notna()
snacks.groupby('has_nutrition')['countries_en'].value_counts(normalize=True).head(10)

has_nutrition  countries_en       
False          United States          0.619821
               France                 0.071395
               United States,World    0.026931
               Netherlands            0.024035
               India                  0.014525
               Australia              0.014029
               Thailand               0.011769
               United Kingdom         0.010844
               Spain                  0.009607
               Germany                0.008903
Name: proportion, dtype: float64

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
import os
print(os.path.exists('/content/drive/MyDrive/market_gap/snacks_raw.parquet'))
!ls -lh /content/drive/MyDrive/market_gap/

True
total 42M
-rw------- 1 root root 42M Jul 17 13:07 snacks_raw.parquet


In [3]:
import pandas as pd
WORK = '/content/drive/MyDrive/market_gap'
snacks = pd.read_parquet(f'{WORK}/snacks_raw.parquet')
snacks['has_nutrition'] = snacks['sugars_100g'].notna() & snacks['proteins_100g'].notna()
print(snacks.shape)

(331907, 15)


In [5]:
snacks['has_nutrition'] = snacks['sugars_100g'].notna() & snacks['proteins_100g'].notna()

In [6]:
pd.crosstab(snacks['countries_en'], snacks['has_nutrition'], normalize='index') \
  .reindex(['France','United States','Italy','Germany','Spain','United Kingdom']) \
  .round(3)

has_nutrition,False,True
countries_en,,
France,0.109,0.891
United States,0.947,0.053
Italy,0.050,0.950
Germany,0.075,0.925
Spain,0.082,0.918
United Kingdom,0.137,0.863


In [7]:
EU = ['France','Italy','Germany','Spain','United Kingdom','Belgium','Switzerland','Netherlands']
pattern = '|'.join(EU)

eu = snacks[snacks['countries_en'].fillna('').str.contains(pattern, case=False, na=False)].copy()
print("EU rows:", len(eu))
print("Coverage:", eu['has_nutrition'].mean().round(3))

eu = eu[eu['has_nutrition']]
print("After dropna:", len(eu))

EU rows: 193086
Coverage: 0.896
After dropna: 172938


In [8]:
audit = []
def log(step, df):
    audit.append({'step': step, 'rows': len(df)})
    print(f"{step:<38} {len(df):>8,}")

log('Raw snacks (global)', snacks)
log('EU markets only', snacks[snacks['countries_en'].fillna('').str.contains(pattern, case=False, na=False)])
log('Nutrition data present', eu)

Raw snacks (global)                     331,907
EU markets only                         193,086
Nutrition data present                  172,938


In [10]:
eu[['sugars_100g','proteins_100g','fat_100g','fiber_100g','salt_100g','energy-kcal_100g']].dtypes

,0
sugars_100g,float64
proteins_100g,float64
fat_100g,object
fiber_100g,float64
salt_100g,float64
energy-kcal_100g,float64


In [11]:
NUM_COLS = ['sugars_100g','proteins_100g','fat_100g','saturated-fat_100g',
            'fiber_100g','salt_100g','energy-kcal_100g']

for c in NUM_COLS:
    if c in eu.columns:
        eu[c] = pd.to_numeric(eu[c], errors='coerce')

print(eu[NUM_COLS].dtypes)
print("\nNulls introduced by coercion:")
print(eu[NUM_COLS].isna().sum())

sugars_100g           float64
proteins_100g         float64
fat_100g              float64
saturated-fat_100g    float64
fiber_100g            float64
salt_100g             float64
energy-kcal_100g      float64
dtype: object

Nulls introduced by coercion:
sugars_100g               0
proteins_100g             0
fat_100g                219
saturated-fat_100g      609
fiber_100g            77327
salt_100g              7515
energy-kcal_100g         35
dtype: int64


In [12]:
NUTRIENTS = ['sugars_100g','proteins_100g','fat_100g','fiber_100g','salt_100g']

for c in NUTRIENTS:
    if c in eu.columns:
        eu = eu[eu[c].isna() | eu[c].between(0, 100)]
log('Nutrients within 0-100g', eu)

eu = eu[eu[['sugars_100g','proteins_100g','fat_100g']].sum(axis=1) <= 100]
log('Macros sum <= 100g', eu)

if 'energy-kcal_100g' in eu.columns:
    eu = eu[eu['energy-kcal_100g'].isna() | eu['energy-kcal_100g'].between(0, 900)]
log('Energy 0-900 kcal', eu)

eu = eu.drop_duplicates(subset=['code'])
log('Deduplicated on barcode', eu)

Nutrients within 0-100g                 172,794
Macros sum <= 100g                      172,689
Energy 0-900 kcal                       172,473
Deduplicated on barcode                 172,473


In [13]:
pre = snacks[snacks['has_nutrition']].copy()
for c in NUM_COLS:
    if c in pre.columns:
        pre[c] = pd.to_numeric(pre[c], errors='coerce')

bad = pre[~pre[NUTRIENTS].apply(lambda s: s.isna() | s.between(0,100)).all(axis=1)]
print(len(bad), "rows with impossible values")
bad[['product_name','sugars_100g','proteins_100g','fat_100g']].sort_values('sugars_100g', ascending=False).head(10)

278 rows with impossible values


,product_name,sugars_100g,proteins_100g,fat_100g
126576,Banan's,74000.000000,2.800000,0.500000
21573,Starburst Original,1600.000000,0.000000,250.000000
183787,Haribo Banana,1283.333333,48.333333,8.333333
327958,Nesquik Chocolate Milk Flavouring,727.272727,41.322314,33.057851
250108,lindt,587.500000,82.500000,462.500000
115255,Marzipan Taler,517.000000,5.400000,19.700000
312666,Cin,360.980000,4.800000,14.000000
328082,Caramello Koala,355.555556,44.444444,173.333333
262427,None,335.000000,0.000000,0.000000
97352,Probiotic Mango-Peach Yoggies,275.000000,0.000000,62.500000


In [14]:
(eu['product_name'].isin(['None','nan','']) | eu['product_name'].isna()).sum()

np.int64(3697)

In [15]:
JUNK = ['cooking-helpers','baking-mixes','cake-mixes','dessert-mixes',
        'pastry-helpers','condiments','sauces']
mask = eu['categories_tags'].fillna('').str.contains('|'.join(JUNK), case=False)
print(mask.sum(), "rows flagged as non-snack")
eu[mask][['product_name','categories_tags','sugars_100g','proteins_100g']].head(15)

1160 rows flagged as non-snack


,product_name,categories_tags,sugars_100g,proteins_100g
1117,CHIA SEED MUFFIN MIX CHOCOLATE,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",4.300000,32.500000
1736,English breakfast muffin mix,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",5.200000,11.500000
4867,Chocolate chocolate chip muffin mix,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",42.857143,7.142857
4881,Hush Puppy Mix with Onion Flavor,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",5.560000,8.330000
4888,BANANA NUT MUFFIN MIX,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",34.700000,5.560000
5833,Mott's Applesauce Apple,"en:plant-based-foods-and-beverages,en:plant-ba...",8.890000,0.000000
6522,Betty Crocker Gluten Free Chocolate Brownie Mix,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",64.290000,3.570000
6547,Betty Crocker Fudge Brownie Mix,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",62.500000,3.125000
6551,Chocolate chip cookie mix,"en:snacks,en:sweet-snacks,en:cooking-helpers,e...",52.173913,4.347826
6613,Super moist cake mix white pudding in the mix ...,"en:snacks,en:sweet-snacks,en:biscuits-and-cake...",19.000000,2.000000


In [16]:
eu = eu[~(eu['product_name'].isin(['None','nan','']) | eu['product_name'].isna())]
log('Valid product names', eu)

Valid product names                     168,776


In [17]:
JUNK = {'en:cooking-helpers','en:baking-mixes','en:cake-mixes','en:dessert-mixes',
        'en:pastry-helpers','en:condiments','en:sauces'}

def is_junk(tags):
    return bool(JUNK & {t.strip() for t in (tags or '').split(',')})

mask = eu['categories_tags'].fillna('').apply(is_junk)
print(mask.sum(), "rows flagged")

1119 rows flagged


In [18]:
eu = eu[~mask]
log('Non-snack products removed', eu)
pd.DataFrame(audit)

Non-snack products removed              167,657


,step,rows
0,Raw snacks (global),331907
1,EU markets only,193086
2,Nutrition data present,172938
3,Nutrients within 0-100g,172794
4,Macros sum <= 100g,172689
5,Energy 0-900 kcal,172473
6,Deduplicated on barcode,172473
7,Valid product names,168776
8,Non-snack products removed,167657


In [19]:
eu.to_parquet(f'{WORK}/snacks_eu_clean.parquet', index=False)
pd.DataFrame(audit).to_csv(f'{WORK}/cleaning_audit.csv', index=False)
!ls -lh "{WORK}/"

total 64M
-rw------- 1 root root 264 Jul 17 17:16 cleaning_audit.csv
-rw------- 1 root root 22M Jul 17 17:16 snacks_eu_clean.parquet
-rw------- 1 root root 42M Jul 17 13:07 snacks_raw.parquet


In [20]:
from collections import Counter
tags = Counter()
for t in eu['categories_tags'].dropna():
    tags.update(x.strip() for x in t.split(','))
pd.Series(dict(tags.most_common(40)))

,0
en:snacks,166883
en:sweet-snacks,137034
en:biscuits-and-cakes,56344
en:confectioneries,42602
en:cocoa-and-its-products,33579
en:biscuits,30089
en:salty-snacks,25326
en:chocolates,22142
en:appetizers,20607
en:biscuits-and-crackers,17975


In [21]:
BUCKETS = [
    ('Bars & Protein Snacks', {'en:cereal-bars','en:bars','en:protein-bars',
                               'en:energy-bars','en:fruit-bars'}),
    ('Nuts & Seeds',          {'en:nuts-and-their-products','en:nuts','en:almonds',
                               'en:peanuts','en:cashew-nuts','en:seeds','en:pistachios'}),
    ('Dried Fruit',           {'en:dried-fruits','en:raisins','en:dates','en:dried-apricots'}),
    ('Chips & Savoury',       {'en:crisps','en:potato-crisps','en:chips-and-fries',
                               'en:crackers-appetizers','en:corn-chips','en:extruded-snacks',
                               'en:popcorn','en:pretzels','en:salty-snacks'}),
    ('Biscuits & Cookies',    {'en:biscuits','en:chocolate-biscuits','en:shortbread-cookies',
                               'en:biscuits-and-crackers','en:cookies'}),
    ('Cakes & Pastries',      {'en:cakes','en:chocolate-cakes','en:viennoiseries','en:pastries',
                               'en:brioches','en:madeleines','en:panettone',
                               'en:sweet-pastries-and-pies'}),
    ('Chocolate',             {'en:dark-chocolates','en:milk-chocolates','en:chocolates',
                               'en:chocolate-candies','en:white-chocolates',
                               'en:cocoa-and-its-products'}),
    ('Confectionery',         {'en:candies','en:bonbons','en:gummies','en:marshmallows',
                               'en:confectioneries'}),
]

FALLBACK = [('Other Sweet', {'en:sweet-snacks'}), ('Other Savoury', {'en:salty-snacks'})]

def assign(tags):
    t = {x.strip() for x in (tags or '').split(',')}
    for name, keys in BUCKETS + FALLBACK:
        if t & keys:
            return name
    return 'Other'

eu['primary_category'] = eu['categories_tags'].apply(assign)
eu['primary_category'].value_counts(dropna=False)

,count
primary_category,
Cakes & Pastries,32912
Chocolate,30544
Biscuits & Cookies,29432
Confectionery,26983
Chips & Savoury,24467
Bars & Protein Snacks,8632
Other Sweet,6667
Other,4705
Nuts & Seeds,2954


In [22]:
print("Other rate:", (eu['primary_category'] == 'Other').mean().round(3))
eu[eu['primary_category'] == 'Other']['categories_tags'].str.split(',').explode() \
  .str.strip().value_counts().head(20)

Other rate: 0.028


,count
categories_tags,
en:snacks,3931
en:plant-based-foods-and-beverages,1311
en:plant-based-foods,1273
en:cereals-and-potatoes,791
en:baby-foods,735
en:snacks-and-desserts-for-babies,733
en:taralli,476
en:extruded-crispbreads,440
en:desserts,400


In [23]:
eu[eu['categories_tags'].str.contains('en:chocolate-biscuits', na=False)]['primary_category'].value_counts()

,count
primary_category,
Biscuits & Cookies,6709
Bars & Protein Snacks,126
Chips & Savoury,4
Nuts & Seeds,3


In [24]:
BABY = {'en:baby-foods','en:snacks-and-desserts-for-babies','en:baby-fruit-desserts',
        'en:from-6-months','en:dairy-dessert-for-baby','en:baby-snacks'}
mask_baby = eu['categories_tags'].fillna('').apply(
    lambda t: bool(BABY & {x.strip() for x in t.split(',')}))
print(mask_baby.sum(), "baby-food rows")
eu = eu[~mask_baby]
log('Baby food removed', eu)

846 baby-food rows
Baby food removed                       166,811


In [25]:
eu.to_parquet(f'{WORK}/snacks_eu_clean.parquet', index=False)

In [26]:
# in BUCKETS, Chips & Savoury set — add:
'en:taralli','it:taralli','en:extruded-crispbreads','en:chips','en:crispbreads'

eu['primary_category'] = eu['categories_tags'].apply(assign)
print("Other rate:", (eu['primary_category']=='Other').mean().round(3))
eu['primary_category'].value_counts()

Other rate: 0.024


,count
primary_category,
Cakes & Pastries,32909
Chocolate,30541
Biscuits & Cookies,29354
Confectionery,26982
Chips & Savoury,24455
Bars & Protein Snacks,8621
Other Sweet,6664
Other,3970
Nuts & Seeds,2954


In [27]:
pd.DataFrame(audit).to_csv(f'{WORK}/cleaning_audit.csv', index=False)

In [28]:
stats = eu.groupby('primary_category').agg(
    products=('code','size'),
    median_sugar=('sugars_100g','median'),
    median_protein=('proteins_100g','median'),
).round(1).sort_values('products', ascending=False)
stats

,products,median_sugar,median_protein
primary_category,,,
Cakes & Pastries,32909,25.0,6.0
Chocolate,30541,45.8,6.8
Biscuits & Cookies,29354,28.0,6.5
Confectionery,26982,53.0,3.0
Chips & Savoury,24455,2.3,6.8
Bars & Protein Snacks,8621,29.2,7.8
Other Sweet,6664,34.2,5.7
Other,3970,3.7,9.0
Nuts & Seeds,2954,7.6,18.0


In [31]:
import numpy as np
SUGAR_MAX, PROTEIN_MIN = 10, 10

eu['quadrant'] = np.select(
    [(eu.sugars_100g < SUGAR_MAX) & (eu.proteins_100g >= PROTEIN_MIN),
     (eu.sugars_100g >= SUGAR_MAX) & (eu.proteins_100g >= PROTEIN_MIN),
     (eu.sugars_100g < SUGAR_MAX) & (eu.proteins_100g < PROTEIN_MIN)],
    ['Blue Ocean (Low Sugar, High Protein)','High Sugar + High Protein','Low Sugar, Low Protein'],
    default='Sugar Trap (High Sugar, Low Protein)')

eu['quadrant'].value_counts(normalize=True).round(3)

,proportion
quadrant,
"Sugar Trap (High Sugar, Low Protein)",0.707
"Low Sugar, Low Protein",0.167
"Blue Ocean (Low Sugar, High Protein)",0.068
High Sugar + High Protein,0.059


In [32]:
bars = eu[eu.primary_category=='Bars & Protein Snacks']
print(bars.proteins_100g.describe().round(1))
print("Bars in blue ocean:", ((bars.sugars_100g<10)&(bars.proteins_100g>=10)).mean().round(3))

count    8621.0
mean       11.1
std         8.9
min         0.0
25%         5.8
50%         7.8
75%        13.0
max        76.0
Name: proteins_100g, dtype: float64
Bars in blue ocean: 0.091


In [33]:
eu.groupby('primary_category')['proteins_100g'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
primary_category,,,,,,,,
Bars & Protein Snacks,8621.0,11.1,8.9,0.0,5.8,7.8,13.0,76.0
Biscuits & Cookies,29354.0,6.8,3.2,0.0,5.5,6.5,7.6,85.0
Cakes & Pastries,32909.0,6.1,2.7,0.0,4.7,6.0,7.2,80.0
Chips & Savoury,24455.0,8.3,5.9,0.0,5.7,6.8,9.9,78.2
Chocolate,30541.0,6.9,2.7,0.0,5.5,6.8,8.2,75.8
Confectionery,26982.0,3.9,4.5,0.0,0.2,3.0,6.2,97.0
Dried Fruit,361.0,5.6,5.5,0.0,1.9,2.9,9.0,24.2
Nuts & Seeds,2954.0,16.0,8.5,0.0,9.5,18.0,22.6,58.0
Other,3970.0,10.3,7.4,0.0,6.3,9.0,12.0,77.0


In [34]:
ct = pd.crosstab(eu.primary_category, eu.quadrant, normalize='index').round(3)
ct[['Blue Ocean (Low Sugar, High Protein)']].sort_values(
    'Blue Ocean (Low Sugar, High Protein)', ascending=False)

pd.crosstab(eu.primary_category, eu.quadrant)   # absolute counts too

quadrant,"Blue Ocean (Low Sugar, High Protein)",High Sugar + High Protein,"Low Sugar, Low Protein","Sugar Trap (High Sugar, Low Protein)"
primary_category,,,,
Bars & Protein Snacks,788,2055,320,5458
Biscuits & Cookies,464,1240,1472,26178
Cakes & Pastries,433,693,2555,29228
Chips & Savoury,5753,309,16724,1669
Chocolate,549,2132,890,26970
Confectionery,298,2207,3869,20608
Dried Fruit,14,64,35,248
Nuts & Seeds,1655,517,125,657
Other,1211,333,1494,932


In [35]:
for s, p in [(10,10), (12,8), (8,12), (5,15)]:
    q = ((eu.sugars_100g < s) & (eu.proteins_100g >= p))
    top = eu[q].primary_category.value_counts(normalize=True).head(2)
    print(f"sugar<{s}, protein>={p}: {q.mean():.3f} blue ocean | top: {dict(top.round(2))}")

sugar<10, protein>=10: 0.068 blue ocean | top: {'Chips & Savoury': np.float64(0.51), 'Nuts & Seeds': np.float64(0.15)}
sugar<12, protein>=8: 0.105 blue ocean | top: {'Chips & Savoury': np.float64(0.5), 'Other': np.float64(0.11)}
sugar<8, protein>=12: 0.046 blue ocean | top: {'Chips & Savoury': np.float64(0.46), 'Nuts & Seeds': np.float64(0.2)}
sugar<5, protein>=15: 0.019 blue ocean | top: {'Chips & Savoury': np.float64(0.41), 'Bars & Protein Snacks': np.float64(0.19)}


In [36]:
bo_chips = eu[(eu.primary_category=='Chips & Savoury') &
              (eu.quadrant=='Blue Ocean (Low Sugar, High Protein)')]
print(bo_chips.brands.value_counts().head(15))
print(bo_chips[['product_name','proteins_100g','sugars_100g']].sample(15, random_state=1))

brands
Carrefour       78
La Mole         70
Picard          67
Auchan          56
U               51
Lorenz          49
Boehli          46
Thiriet         42
Caputo          40
Belin           39
Galbusera       36
Casino          33
Leader Price    31
Snack Day       31
Zorzi           27
Name: count, dtype: int64
                                             product_name  proteins_100g  \
316028                                           Scrocchi           10.0   
223448                                         Bake rolls           12.0   
176156                             NoCOé Crackers Romarin           15.0   
145343           Prefou extra brie & truffe blanche d'été           10.8   
151254                           Crackers aperitif nature           11.0   
198179                                       Lentil Chips           10.4   
128960                          Pop Corn Apéro, Goût Salé           11.0   
190575                    Toasty - Southern Fried Chicken           18.0  

In [37]:
print(bo_chips[['proteins_100g','sugars_100g']].describe().round(1))

       proteins_100g  sugars_100g
count         5753.0       5753.0
mean            15.0          2.7
std              8.2          2.0
min             10.0          0.0
25%             11.0          1.3
50%             12.3          2.3
75%             15.0          3.6
max             78.2          9.9
